# Stage 5 — extract

Re-create this stage's script with Gemini's help. The cells below give you the spec, the seed, the gotchas, and a verification step. The implementation itself is yours to write.


## 1. Setup

Every cell in this section is idempotent and safe to re-run. If you opened this notebook fresh (without running Stage 0 first in the same runtime), run all of them now.


### 1a. Clone the repo and `cd` into it


In [ ]:
# Bootstrap: clone the workshop repo into /content and cd into it.
# Idempotent — safe to re-run.
import os, subprocess, sys
REPO_DIR = "/content/ar-bic-2026-workshop"
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/jayprimer/ar-bic-2026-workshop.git", REPO_DIR],
        check=True,
    )
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())


### 1b. Install dependencies

Python (`openai`) and the Node CLI `@llamaindex/liteparse`. First run takes ~30s; re-runs are near-instant.


In [ ]:
# Install dependencies. Idempotent (pip skips already-installed; npm re-link is cheap).
# liteparse only matters for Stage 4 but installing it everywhere keeps each
# notebook self-contained, which is the whole point of re-running this cell.
!pip install -q -r requirements.txt
!npm install -g @llamaindex/liteparse 2>&1 | tail -3


### 1c. Bridge your OpenAI key

Add `OPENAI_API_KEY` in Colab's Secrets panel (key icon, left sidebar) and toggle notebook access first.


In [ ]:
# OpenAI key bridge: Colab's userdata.get() does NOT populate os.environ,
# but our scripts read os.environ["OPENAI_API_KEY"]. Bridge it once.
# Add the key in Colab via the left sidebar → "Secrets" (key icon) → name it OPENAI_API_KEY.
import os
try:
    from google.colab import userdata
    key = userdata.get("OPENAI_API_KEY")
    if key:
        os.environ["OPENAI_API_KEY"] = key
        print("OPENAI_API_KEY set in os.environ")
    else:
        print("WARNING: OPENAI_API_KEY secret is empty — Stage 2/5 and *_llm.py evals will fail")
except Exception as e:
    print("Not running in Colab or userdata unavailable; set OPENAI_API_KEY yourself.")
    print("Detail:", e)


### 1d. Stage the bundle-shipped configs


In [ ]:
# Copy bundle-shipped configs into the directories each stage script expects.
# Each stage's input.txt / criteria.txt / schema.json lives under configs/
# in the repo; the actual scripts read them relative to cwd.
import os, shutil
os.makedirs("stage_01", exist_ok=True)
os.makedirs("stage_02", exist_ok=True)
shutil.copy("configs/stage_01_input.txt",    "stage_01/input.txt")
shutil.copy("configs/stage_02_input.txt",    "stage_02/input.txt")
shutil.copy("configs/stage_02_criteria.txt", "stage_02/criteria.txt")
shutil.copy("configs/schema.json",           "schema.json")
print("configs staged")


### 1e. Load prior stages' reference outputs

Stage 5 reads outputs from earlier stages. Each Colab notebook gets its own runtime, so work done in another notebook is not visible here. This cell seeds `stage_01..stage_04/data/` from the canonical reference run so Stage 5 has inputs to work with.


In [ ]:
# Load prior stages' reference outputs as inputs for Stage 5.
# Each Colab notebook opens with a fresh runtime, so any work done in a
# Stage <5 notebook in a DIFFERENT runtime is not visible here.
# This cell makes the stage runnable in isolation against the canonical
# reference run. If you re-run an earlier stage IN THIS runtime, your
# output replaces these reference files (cwd is /content/...).
import os, shutil, glob
for n in range(1, 5):
    dst = f"stage_0{n}/data"
    src = f"reference_outputs/stage_0{n}/data"
    if not os.path.isdir(src):
        continue
    os.makedirs(dst, exist_ok=True)
    # Only seed if the participant hasn't produced anything for this stage
    # in the current runtime — otherwise we'd clobber their work.
    if any(os.scandir(dst)):
        print(f"skip stage_0{n} — already has files (keeping your work)")
        continue
    for src_file in glob.glob(f"{src}/*"):
        shutil.copy(src_file, dst)
    print(f"seeded stage_0{n}/data from reference_outputs")


## 2. Spec — paste this into Gemini

Open the Gemini side panel in Colab (sparkles icon, top right) and paste the block below as your prompt. Then iterate.

```
For every `stage_04/data/*.md`, call OpenAI gpt-5.4-nano to extract a
JSON record matching `schema.json` (located at repo root). Write to
`stage_05/data/<stem>.json` (one file per paper).

The schema is a TYPE CONTRACT, not literal values. Each field's value
is the TYPE of the data to extract, not the type label itself.

After the LLM returns:
  1. Force-overwrite `pmid` and `source_type` from the
     `stage_03/data/fetched.json` mapping — never trust the LLM for IDs.
  2. Run a cheap hallucination check on `concurrent_nam`: any value
     whose substantive tokens (>4 chars) don't appear in the source
     text gets demoted to `nams_discussed` (keeps it visible for
     review but out of the structured arm).
```


## 3. Gotchas Gemini probably won't know

Copy any that apply into Gemini if it goes off-track:

- **No regex fallback for this stage.** It hard-fails without
  `OPENAI_API_KEY`. (`assert os.environ.get("OPENAI_API_KEY")`)
- **JSON mode required.** `response_format={"type": "json_object"}`
  and `temperature=0`.
- **schema.json lives at the REPO ROOT**, not under `configs/`, after
  Stage 0's setup cell copies it.
- **PMC stem → PMID is one-to-many-ish.** Build the stem→pmid map
  from `stage_03/data/fetched.json` and inject the pmid into the
  prompt so the LLM doesn't invent one.
- **Source type matters.** Tag each record `fulltext` vs
  `abstract-only` based on the Stage 3 path extension. The LLM should
  return `[]` for `animal_arms` rather than guessing from a brief
  PubMed abstract.


## 4. Seed — a few lines to anchor Gemini in the right direction


In [ ]:
import glob, json, os, time
assert os.environ.get("OPENAI_API_KEY"), \
    "extraction needs OPENAI_API_KEY (no regex fallback)"
from openai import OpenAI

STAGE = "stage_05"
DATA = f"{STAGE}/data"
IN_DATA = "stage_04/data"
os.makedirs(DATA, exist_ok=True)

with open("schema.json") as f:
    SCHEMA = f.read()


## 5. Your implementation

Drive Gemini to fill this in. Iterate until the verification cell below passes.


In [ ]:
# TODO: implement Stage 5 here.
# Read the spec above. Use the seed cell's imports.
# When done, run the verification cell next.


## 6. Verify


In [ ]:
import glob, json, os
outs = sorted(glob.glob("stage_05/data/*.json"))
assert outs, "no extractions produced"
for p in outs:
    if os.path.basename(p) in ("eval_script.json", "eval_llm.json", "score.json"):
        continue
    rec = json.load(open(p))
    assert rec.get("pmid"), f"{p}: no pmid"
    assert isinstance(rec.get("animal_arms"), list), f"{p}: animal_arms not list"
print(f"OK — {len(outs)} extractions")


## 7. Run the eval grader

The eval reads only your stage's output and writes `stage_05/eval/eval_*.json` + `score.json`.


In [ ]:
!python eval/eval_05_script.py
# Optional (needs API key, ~$0.10):
# !python eval/eval_05_llm.py


## 8. Stuck? Skip this stage

Copy the reference run's Stage 5 output into place so the next stage's notebook can still run. Use this sparingly — the point of the workshop is to *re-create* each stage.


In [ ]:
import os, shutil, glob
os.makedirs("stage_05/data", exist_ok=True)
for src in glob.glob("reference_outputs/stage_05/data/*.json"):
    shutil.copy(src, "stage_05/data/")
print(f"copied {len(os.listdir('stage_05/data'))} reference extractions")
